# Symmetry Phase 3 - GPT Representations

This notebook trains a tiny GPT on synthetic sequences and analyzes
layer representations with the same symmetry tools as Phase 2.

In [ ]:
import numpy as np

from boardGPT.analysis import (
    PolicyConfig,
    SyntheticGPTTrainConfig,
    SyntheticWorld,
    build_tiny_gpt_for_synthetic,
    extract_mean_token_representations_from_model,
    generate_synthetic_sequences,
    lattice_1d_concept,
    pairwise_distances,
    representation_symmetry_report,
    train_gpt_on_synthetic_sequences,
)

## 1) Generate a periodic 1D world

In [ ]:
concept = lattice_1d_concept(num_tokens=32, periodic=True, period=2.0, name='ring1d')
world = SyntheticWorld(concept=concept)
dist = pairwise_distances(concept)

data_cfg = PolicyConfig(
    length=48,
    num_sequences=1500,
    seed=0,
    transition_kernel='exponential',
    transition_scale=0.18,
    transition_strength=1.0,
    epsilon_uniform=0.03,
)
dataset = generate_synthetic_sequences(world, data_cfg)
print('sequences:', dataset.sequences.shape)

## 2) Train a tiny GPT

In [ ]:
model = build_tiny_gpt_for_synthetic(
    vocab_size=world.vocab_size,
    block_size=dataset.sequences.shape[1] - 1,
    n_layer=2,
    n_head=2,
    n_embd=64,
    dropout=0.0,
    use_flash=False,
)

train_cfg = SyntheticGPTTrainConfig(
    batch_size=64,
    max_steps=120,
    learning_rate=3e-4,
    weight_decay=0.01,
    grad_clip=1.0,
    seed=0,
    device='cpu',
    log_interval=40,
)
losses = train_gpt_on_synthetic_sequences(model, dataset.sequences, train_cfg)
print('initial loss:', losses[0], 'final loss:', losses[-1])

## 3) Extract token representations from layer residuals

In [ ]:
reprs = extract_mean_token_representations_from_model(
    model=model,
    sequences=dataset.sequences,
    layer=1,
    vocab_size=world.vocab_size,
    device='cpu',
    batch_size=128,
)
print('representations:', reprs.shape)

## 4) Run symmetry report on learned representation geometry

In [ ]:
report = representation_symmetry_report(
    representations=reprs,
    distances=dist,
    pca_rank=4,
    num_bins=20,
)

print('symmetry R^2:', report.symmetry.r2)
print('symmetry MSE:', report.symmetry.mse)
print('top PCA eigvals:', report.bundle.pca_eigvals[:4])
print('first 5 token coords (PC1,PC2):')
print(report.bundle.pca_coords[:5, :2])